<a href="https://colab.research.google.com/github/ridanoor0110/DSC_110/blob/main/week8_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [297]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [298]:
df = pd.read_csv('https://raw.githubusercontent.com/osoliman/DSC110/refs/heads/main/Datasets/nhanes_final.csv')

In [299]:
df.head()

,id,gender,age,race_ethnicity,education,bmi,height_cm,systolic_bp,has_diabetes,hba1c,fasting_glucose
0,130378.0,1.0,43.0,6.0,5.0,27.0,179.5,135.0,2.0,5.6,113.0
1,130379.0,1.0,66.0,3.0,5.0,33.5,174.2,121.0,2.0,5.6,99.0
2,130380.0,2.0,44.0,2.0,3.0,29.7,152.9,111.0,1.0,6.2,156.0
3,130381.0,2.0,5.0,7.0,NaN,23.8,120.1,NaN,2.0,NaN,NaN
4,130382.0,1.0,2.0,3.0,NaN,NaN,NaN,NaN,2.0,NaN,NaN


Step 1: Filtering

In [300]:
#Removing all data values for patients that are under 18
df = df[df['age'] >= 18]

#verify by printing the minimum age of our patients in this new data set
df['age'].min()

18.0

Step 2: Drop columns/fields

In [301]:
df = df.drop(columns=['id', 'hba1c', 'fasting_glucose'])
#remove the fields id, hba1c and fasting glucose

In [302]:
df.head()
#prints to confirm that the fields have been removed

,gender,age,race_ethnicity,education,bmi,height_cm,systolic_bp,has_diabetes
0,1.0,43.0,6.0,5.0,27.0,179.5,135.0,2.0
1,1.0,66.0,3.0,5.0,33.5,174.2,121.0,2.0
2,2.0,44.0,2.0,3.0,29.7,152.9,111.0,1.0
6,1.0,43.0,1.0,2.0,NaN,NaN,NaN,2.0
7,2.0,65.0,3.0,3.0,NaN,NaN,NaN,2.0


Step 3: Mapping labels: gender, education, race

In [303]:
#create dictionaries
gendermap = {1.0: 'Male', 2.0: 'Female'}
df['gender'] = df['gender'].map(gendermap)
#this applies the dictionary we created for gender onto our data set.

#repeat for education
educationmap = {
    1.0: 'Less than 9th',
    2.0: '9th - 11th grade',
    3.0: 'High school',
    4.0: 'Some college',
    5.0: 'College grad',
    9.0: 'Do not know'

}
df['education'] = df['education'].map(educationmap)
#applies the dictionary we created for education onto our data set

#verify it worked
df['education'].value_counts()

#finally, for race/ethnicity mapping
racemap = {
    1.0: 'Mexican American',
    2.0: 'Other Hispanic',
    3.0: 'White',
    4.0: 'Black',
    6.0: 'Asian',
    7.0: 'Other'
}
df['race_ethnicity'] = df['race_ethnicity'].map(racemap)

Step 4: Missing values

In [304]:
df.isna().sum()

,0
gender,0
age,0
race_ethnicity,0
education,359
bmi,1918
height_cm,1891
systolic_bp,2030
has_diabetes,3


In [305]:
df['bmi'] = df['bmi'].fillna(df['bmi'].mean())

In [306]:
#loops can condense writing code separately for each category into one block of code that functionally achieves the same purpose
for col in ['bmi', 'height_cm', 'systolic_bp']:
  df[col] = df[col].fillna(df[col].mean)
  #if any values are missing it will be replaced with the mean/average value

In [307]:
#verify the above loop and ensure missing values are fixed
df.isna().sum()

,0
gender,0
age,0
race_ethnicity,0
education,359
bmi,0
height_cm,0
systolic_bp,0
has_diabetes,3


In [308]:
#removing rows where education is missing
df = df.dropna(subset=['education'])

df.isna().sum()

,0
gender,0
age,0
race_ethnicity,0
education,0
bmi,0
height_cm,0
systolic_bp,0
has_diabetes,0


Step 5: Definte Target

In [309]:
#The target is what we are trying to predict. in this case, it is diabetes. we will define diabetes as our target.
df = df[df['has_diabetes'].isin([1.0, 2.0])].copy()
#firstly we include rows that have or do not (yes/no), if any of them are borderline/missing we must remove them.
df['has_diabetes'] = df['has_diabetes'].map({1.0: 1, 2.0: 0})

In [310]:
df['has_diabetes'].value_counts()

,count
has_diabetes,
0,6455
1,1068


Step 6: One-hot encoding

In [311]:
#This converts the categorical columns into dummy variables for analysis, while drop_first=True removes one category from each column to avoid redundant information
df = pd.get_dummies(df, columns=['gender', 'race_ethnicity', 'education'],
                    drop_first=True)
#this function is from pandas (hence, pd)

In [312]:
df.head()

,age,bmi,height_cm,systolic_bp,has_diabetes,gender_Male,race_ethnicity_Black,race_ethnicity_Mexican American,race_ethnicity_Other,race_ethnicity_Other Hispanic,race_ethnicity_White,education_College grad,education_Do not know,education_High school,education_Less than 9th,education_Some college
0,43.0,27.000000,179.5,135.0,0,True,False,False,False,False,False,True,False,False,False,False
1,66.0,33.500000,174.2,121.0,0,True,False,False,False,False,True,True,False,False,False,False
2,44.0,29.700000,152.9,111.0,1,False,False,False,False,True,False,False,False,True,False,False
6,43.0,29.659326,<bound method Series.mean of 0 179.5\n1...,<bound method Series.mean of 0 135.0\n1...,0,True,False,True,False,False,False,False,False,False,False,False
7,65.0,29.659326,<bound method Series.mean of 0 179.5\n1...,<bound method Series.mean of 0 135.0\n1...,0,False,False,False,False,False,True,False,False,True,False,False


Step 7: Scaling numeric fields

In [313]:
#not required for all machine learning algorithms but mostly required for neural network algorithms.
#Standardizes the numerical columns so they have a mean of 0 and a standard deviation of 1.
numeric_cols = ['age', 'bmi']
scaler = StandardScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [314]:
df.head()

,age,bmi,height_cm,systolic_bp,has_diabetes,gender_Male,race_ethnicity_Black,race_ethnicity_Mexican American,race_ethnicity_Other,race_ethnicity_Other Hispanic,race_ethnicity_White,education_College grad,education_Do not know,education_High school,education_Less than 9th,education_Some college
0,-0.589355,-0.425106,179.5,135.0,0,True,False,False,False,False,False,True,False,False,False,False
1,0.719517,0.589889,174.2,121.0,0,True,False,False,False,False,True,True,False,False,False,False
2,-0.532448,-0.003493,152.9,111.0,1,False,False,False,False,True,False,False,False,True,False,False
6,-0.589355,-0.009844,<bound method Series.mean of 0 179.5\n1...,<bound method Series.mean of 0 135.0\n1...,0,True,False,True,False,False,False,False,False,False,False,False
7,0.662610,-0.009844,<bound method Series.mean of 0 179.5\n1...,<bound method Series.mean of 0 135.0\n1...,0,False,False,False,False,False,True,False,False,True,False,False


In [315]:
df[numeric_cols].describe().round(2)
#summary statistics to 2 decimal places
#mean = 0 shows that we have standardized the numerical columns

,age,bmi
count,7523.00,7523.00
mean,0.00,-0.00
std,1.00,1.00
min,-1.90,-2.91
25%,-0.87,-0.61
50%,0.15,-0.01
75%,0.83,0.29
max,1.52,7.04


Step 8: Train test split

In [316]:
X = df.drop(columns=['has_diabetes'])
y = df['has_diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
#Creating 4 files which we will import next time
#Separates the features (X) from the target (y) and splits the data into 80% training and 20% testing sets.
#X is capitalized and y is small

In [317]:
print(X_train.shape, y_train.shape)


(6018, 15) (6018,)


In [318]:
print(X_test.shape, y_test.shape)

(1505, 15) (1505,)
